In [ ]:
import os
import json
import asyncio
import nest_asyncio
from dataclasses import dataclass
from typing import Literal

import anthropic
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()
nest_asyncio.apply()  # lets asyncio.run work inside Jupyter

# Anthropic — kept native (orchestrator uses advanced features)
anthropic_client = anthropic.AsyncAnthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# Everything else via OpenAI-compatible endpoints
openai_client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
)
deepseek_client = AsyncOpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
)
groq_client = AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
gemini_client = AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [ ]:
@dataclass
class Worker:
    name: str
    provider: Literal["openai", "gemini", "groq", "deepseek"]
    model: str
    system_prompt: str

WORKERS: dict[str, Worker] = {
    "market_positioning": Worker(
        name="Market Positioning Analyst",
        provider="openai",
        model="gpt-4o",
        system_prompt=(
            "You are a market research analyst. Given a company and context, "
            "produce a concise analysis of: market position, target customers, "
            "primary competitors, and recent strategic moves. Be specific and factual. "
            "If you're uncertain about a fact, say so rather than guessing. "
            "Respond in under 250 words."
        ),
    ),
    "product_features": Worker(
        name="Product Features Analyst",
        provider="gemini",
        model="gemini-2.5-flash",
        system_prompt=(
            "You are a product analyst. Given a company, extract its core product "
            "capabilities and notable features. Structure output as: Core Product, "
            "Key Features (bulleted), Notable Integrations, Recent Product Updates. "
            "Be concrete. Respond in under 250 words."
        ),
    ),
    "pricing_model": Worker(
        name="Pricing & Business Model Analyst",
        provider="groq",
        model="llama-3.3-70b-versatile",
        system_prompt=(
            "You are a pricing analyst. Given a company, summarize: pricing tiers, "
            "pricing model type (subscription, usage-based, freemium, etc.), "
            "typical customer spend, and unusual pricing decisions. If exact prices "
            "are unknown, describe the model qualitatively. Under 200 words."
        ),
    ),
    "technical_architecture": Worker(
        name="Technical Architecture Analyst",
        provider="deepseek",
        model="deepseek-chat",
        system_prompt=(
            "You are a technical analyst. Given a company, describe: underlying "
            "technology stack, architectural differentiators, performance/scale "
            "characteristics, and any technical moat. Only include what's publicly "
            "knowable. Under 250 words."
        ),
    ),
}

In [ ]:
PROVIDER_CLIENTS = {
    "openai":   openai_client,
    "deepseek": deepseek_client,
    "groq":     groq_client,
    "gemini":   gemini_client,
}

async def call_worker(worker: Worker, task: str) -> str:
    """Dispatch a task to the appropriate provider via OpenAI-compatible API."""
    try:
        client = PROVIDER_CLIENTS[worker.provider]
        resp = await client.chat.completions.create(
            model=worker.model,
            messages=[
                {"role": "system", "content": worker.system_prompt},
                {"role": "user", "content": task},
            ],
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"[Worker {worker.name} failed: {type(e).__name__}: {e}]"

In [ ]:
ORCHESTRATOR_SYSTEM = """You are an orchestrator for a competitive analysis system.

Given a user's analysis request, you must:
1. Identify the companies to analyze (2-5 companies).
2. Select which analysis dimensions are most relevant from this fixed set:
   - market_positioning
   - product_features
   - pricing_model
   - technical_architecture

Choose 3-4 dimensions based on what matters most for this specific query.
For example, a query about enterprise software focuses on pricing and features;
a query about infrastructure focuses on technical architecture.

Respond ONLY with valid JSON in this exact shape:
{
  "companies": ["Company A", "Company B", ...],
  "dimensions": ["dimension_key_1", "dimension_key_2", ...],
  "rationale": "one-sentence explanation of your choices"
}
"""

async def orchestrate(user_query: str) -> dict:
    resp = await anthropic_client.messages.create(
        model="claude-opus-4-5",
        max_tokens=1024,
        system=ORCHESTRATOR_SYSTEM,
        messages=[{"role": "user", "content": user_query}],
    )
    text = resp.content[0].text
    # Strip markdown fences if the model added them
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

    """The orchestrator constrains output to valid JSON — this is how you turn freeform reasoning into structured data you can iterate over. The removeprefix/removesuffix dance handles the case where Claude wraps JSON in markdown fences despite being told not to. Small defensive move that saves debugging time.
Dimensions are a fixed set because workers are predefined. But notice the orchestrator chooses which to use per query — that's the dynamic decomposition."""

In [ ]:
async def analyze(user_query: str) -> dict:
    # Step 1: Plan
    print("🧭 Orchestrator planning...")
    plan = await orchestrate(user_query)
    print(f"   Companies:  {plan['companies']}")
    print(f"   Dimensions: {plan['dimensions']}")
    print(f"   Rationale:  {plan['rationale']}\n")

    # Step 2: Dispatch all (company × dimension) tasks in parallel
    tasks = []
    task_meta = []  # parallel list to track what each task is
    for company in plan["companies"]:
        for dim in plan["dimensions"]:
            worker = WORKERS[dim]
            prompt = f"Analyze: {company}\n\nContext: {user_query}"
            tasks.append(call_worker(worker, prompt))
            task_meta.append((company, dim, worker.name))

    print(f"🔨 Dispatching {len(tasks)} worker calls across providers...\n")
    results = await asyncio.gather(*tasks)

    # Step 3: Organize results
    findings: dict[str, dict[str, str]] = {c: {} for c in plan["companies"]}
    for (company, dim, _), result in zip(task_meta, results):
        findings[company][dim] = result

    # Step 4: Synthesize — back to orchestrator for the final report
    print("🧩 Synthesizing final report...\n")
    synthesis_prompt = f"""Original query: {user_query}

Worker findings (JSON):
{json.dumps(findings, indent=2)}

Write a competitive analysis report. Start with a 2-3 sentence executive summary,
then a comparison organized by dimension (not by company), then end with
'Key Takeaways' — 3-5 bullets. Call out where companies meaningfully differ.
"""
    resp = await anthropic_client.messages.create(
        model="claude-opus-4-5",
        max_tokens=4096,
        messages=[{"role": "user", "content": synthesis_prompt}],
    )
    report = resp.content[0].text

    return {"plan": plan, "findings": findings, "report": report}

    """The synthesis step is the other half of the pattern people sometimes forget: you need the orchestrator (or another synthesizer) to stitch the worker outputs back together. 
    Raw worker outputs are ingredients, not a deliverable."""

In [ ]:
from IPython.display import Markdown, display

query = "Compare the leading enterprise data warehouse providers for a mid-sized SaaS company"

result = await analyze(query)

display(Markdown("## 📊 Final Report\n\n" + result["report"]))

Experiments to try once it's working
Once you've got a baseline run, these will cement the pattern:

Try a query that names specific companies ("Compare Snowflake, Databricks, and BigQuery") vs. one that doesn't ("Who are the leaders in vector databases?"). Watch how the orchestrator handles each — the second requires it to infer the competitor set, which is a much harder planning task.

Time the full run, then try making the worker calls sequential (swap asyncio.gather for a loop). On a 4-company × 4-dimension query that's 16 calls — sequential will be painfully slow, and you'll feel why parallelism matters.

Add a fifth worker using a different model on a provider you're already using, and teach the orchestrator about it by editing ORCHESTRATOR_SYSTEM. This shows how extensible the pattern is: workers are just plug-in specialists.

Model-name sanity check before you run
Model IDs change more often than anything else in these setups. If Cell 6 fails with a "model not found" error, check these two first:

Gemini: if gemini-2.5-flash errors on the compat endpoint, try gemini-2.0-flash.
Groq: if llama-3.3-70b-versatile has been deprecated, their current models page will list the replacement — Groq rotates model availability more than other providers.

Everything else should be stable.